# Load `bronze.sample_orders` from a CSV file

Every run **reads** `files/source/shopflow/sample_orders.csv` from your bucket and **replaces the
whole table** `bronze.sample_orders` with it — a full reload. Edit the CSV (or drop a new one in its
place with **My files**), run all cells, and the table matches the file again.

`iceberg` is your own lakehouse (`__LAKE__`); in SQLPad / Trino / Superset the same table is
`__LAKE__.bronze.sample_orders`.

## 1 · Read the CSV
A **schema** says exactly which columns and types to expect — no guessing (`inferSchema`) and the
same result every run.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

source = "s3a://__BUCKET__/files/source/shopflow/sample_orders.csv"

schema = StructType([
    StructField("order_id",    IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("channel",     StringType()),
    StructField("order_ts",    TimestampType()),
    StructField("status",      StringType()),
    StructField("currency",    StringType()),
    StructField("promo_id",    IntegerType()),
])

df = spark.read.option("header", True).schema(schema).csv(source)
print(df.count(), "rows read from", source)
df.show(5)

## 2 · Overwrite the table
`createOrReplace()` replaces the **entire** table — its rows *and* its schema — in one atomic
step (readers see the old or the new table, never half of each). Iceberg keeps the previous
version as a **snapshot**, so you can still look back (step 4).

In [ ]:
df.writeTo("iceberg.bronze.sample_orders").using("iceberg").createOrReplace()
print("bronze.sample_orders replaced")

## 3 · Check it

In [ ]:
%%sql
SELECT status, count(*) AS orders
FROM iceberg.bronze.sample_orders
GROUP BY status
ORDER BY orders DESC

## 4 · Every run is a new snapshot
Each overwrite adds a row here — the table's history.

In [ ]:
%%sql
SELECT committed_at, snapshot_id, operation
FROM iceberg.bronze.sample_orders.snapshots
ORDER BY committed_at DESC